# Paper alignment — Pingitore et al., *IJC* 404 (2024) 131981

**Scope.** Retrace the *methodological steps* of

> Pingitore et al., "Machine learning to identify a composite indicator to
> predict cardiac death in ischemic heart disease", *International Journal of
> Cardiology* 404 (2024) 131981,

on the project's **strict** cohort, so the thesis results (extension with
thyroid function) become **comparable** with that paper.

The goal is **not** to reproduce the same numbers, but to **retrace the same
steps while being explicit about the differences**. Cohort construction and
feature engineering are reused from the project. Heavy computations live in
`src.alignment.run_alignment` (pre-tuning sensitivity analyses) and
`src.alignment.paper_tuning` (extended paper-like random search), with
fingerprinted caches under `reports/alignment/`.

**Key differences vs the paper** (detailed below):
- Cohort: **strict only**; per scelta progettuale, le morti non-CVD avvenute
  dopo l'orizzonte restano tra i sopravvissuti all'orizzonte.
- CV17 stays at **17 cardiac variables**; **creatinine / eGFR remain EXCLUDED**
  (too many missing). The paper used 18 variables incl. creatinine — documented,
  not added.
- Primary paper-like validation: **60/20/20**, with 5,000-draw random search and
  internal 2-fold CV on training only; the held-out test is never used for
  selection. Existing fixed-parameter A/B results are retained as
  **pre-tuning sensitivity analyses**.
- Missing values: median imputation fitted inside training folds, rather than
  the paper's replacement with zero.
- Optional sampling refinement follows the random search and uses validation
  only; it is never selected on test.
- Thyroid states encoded as **5 dummies** (eutiroideo = reference).
- Horizons: **7 and 10 years**.


In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from configs.config import RANDOM_STATE
from src.alignment.cohorts10 import build_strict_cohorts, summarize
from src.alignment.features_align import (
    describe_feature_sets, extract_Xy_align, get_align_feature_set,
)
from src.alignment.holdout_split import split_60_20_20, describe_split
from src.alignment.ensemble import ALIGN_MODELS, ENSEMBLE_MEMBERS, ENSEMBLE_NAMES
from src.alignment import run_alignment as RA
from src.alignment.paper_tuning import (
    PAPER_PROFILES,
    PaperTuningConfig,
    load_cached_paper_tuning,
    paper_sampling_candidates,
    paper_search_space_summary,
    run_paper_tuning,
)
from src.alignment.clinical_utility import (
    calibration_curve_table,
    calibration_summary,
    clinical_impact_table,
    decision_curve,
)

# ── shared parameters ──
SEED      = RANDOM_STATE
HORIZONS  = (7, 10)
N_BOOT    = 1000

# Fixed-parameter sensitivity caches. Set FORCE=True only to recompute them.
FORCE = False

# Extended tuning is intentionally opt-in: the default path below only reads a
# matching cache and can never start the 5,000-draw searches accidentally.
RUN_EXTENDED_TUNING = False
PAPER_TUNING_CONFIG = PaperTuningConfig(
    n_iter=5000,
    cv=2,
    scoring="f1_macro",
    n_jobs=-1,
    seed=SEED,
    profile="primary",
    threshold=0.5,
    sampling_refinement=True,
    sampling_repeats=5,
    bootstrap_repeats=N_BOOT,
)

# Prespecified decision-analysis scenarios. These are not learned or optimised
# on the test set and are not asserted to be validated clinical cut-offs.
CLINICAL_RISK_THRESHOLDS = np.array([0.10, 0.20, 0.30, 0.40, 0.50])
CALIBRATION_BINS = 10

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("Models reported:", ALIGN_MODELS)
print("Ensemble members (mean of probabilities):", ENSEMBLE_MEMBERS)
print("Extended tuning auto-run:", RUN_EXTENDED_TUNING)


Models reported: ['LogisticRegression', 'RandomForest', 'AdaBoost', 'SVC', 'KNeighbors', 'MLP', 'GradientBoosting', 'XGBoost', 'ENSEMBLE', 'ENS_VOTE_XGB', 'ENS_VOTE_BOOST', 'ENS_VOTE_DIVERSE', 'ENS_VOTE_WIDE', 'ENS_STACK', 'ENS_STACK_GB']
Ensemble members (mean of probabilities): ['LogisticRegression', 'RandomForest', 'AdaBoost']
Extended tuning auto-run: False


## 0. Cohorts and feature sets (verification)

We rebuild the **strict** cohort at **7 and 10 years** with the *same logic* as
`build_dataset` (only the horizon changes). The 7-year cohort is asserted to be
identical to the saved `cohort_strict.parquet`. For every feature set we print
the exact feature list used in this notebook.

In [2]:
cohorts = build_strict_cohorts(HORIZONS)

# Verify the rebuilt 7y strict cohort matches the saved one.
saved = pd.read_parquet(ROOT / "data" / "processed" / "cohort_strict.parquet")
assert len(cohorts[7]) == len(saved), "7y N mismatch vs saved strict cohort"
assert int(cohorts[7]["y7"].sum()) == int(saved["y7"].sum()), "7y events mismatch"
print("OK: rebuilt 7y strict cohort matches saved cohort_strict.parquet")

summ = pd.DataFrame([summarize(cohorts[h], h) for h in HORIZONS])
display(Markdown("**Strict cohort — N, events (CVD death), prevalence**"))
display(summ)


OK: rebuilt 7y strict cohort matches saved cohort_strict.parquet


**Strict cohort — N, events (CVD death), prevalence**

,horizon_years,N,n_events_cvd,n_survivors,prevalence
0,7,4390,843,3547,0.192
1,10,2614,962,1652,0.368


In [3]:
display(Markdown("**Feature sets used in this notebook** "
                 "(5-dummy thyroid states, eutiroideo = reference)"))
display(describe_feature_sets())


**Feature sets used in this notebook** (5-dummy thyroid states, eutiroideo = reference)

,feature_set,n_features,features
0,CV17,17,"Gender, Age, Angina, Previous_CABG, Previous_P..."
1,CV17_THY_CONT_STATES,25,"Gender, Age, Angina, Previous_CABG, Previous_P..."
2,CV17_THY_ABNORMAL_BIN,18,"Gender, Age, Angina, Previous_CABG, Previous_P..."
3,CV17_THY_STATE_ORD,18,"Gender, Age, Angina, Previous_CABG, Previous_P..."
4,CV17_THY_CONT,20,"Gender, Age, Angina, Previous_CABG, Previous_P..."
5,CV17_THY_STATES,22,"Gender, Age, Angina, Previous_CABG, Previous_P..."
6,CV17_THY_CONT_STATES_RATIO,26,"Gender, Age, Angina, Previous_CABG, Previous_P..."
7,CV17_THY_CONT_RATIO,21,"Gender, Age, Angina, Previous_CABG, Previous_P..."


In [4]:
# Illustrate the 60/20/20 stratified split (scheme B) on CV17 @ 7y.
Xb, yb = extract_Xy_align(cohorts[7], "CV17", "y7")
itr, iva, ite = split_60_20_20(yb, seed=SEED)
display(Markdown("**Scheme B — 60/20/20 stratified split (CV17 @ 7y)**"))
display(describe_split(yb, itr, iva, ite))


**Scheme B — 60/20/20 stratified split (CV17 @ 7y)**

,part,n,fraction,n_pos,prevalence
0,train,2634,0.6,506,0.1921
1,val,878,0.2,169,0.1925
2,test,878,0.2,168,0.1913


## 1. Extended random search (Pingitore-like) — primary scheme B

For each model, feature set and horizon in the `primary` profile:

1. stratified 60/20/20 split shared by CV17 and CV17_THY_CONT_STATES;
2. `SimpleImputer(median) → StandardScaler → classifier`;
3. 5,000-draw `RandomizedSearchCV`, F1-macro, two-fold CV on the **60% train
   only**;
4. optional paper sampling refinement for LR/RF/AdaBoost, selected on the
   **20% validation only**;
5. fixed LR+RF+AdaBoost probability average and one-shot evaluation on the
   untouched **20% test**, threshold 0.5;
6. paired bootstrap Δ F1-macro/Δ AUROC for CV17_THY_CONT_STATES vs CV17 on identical test
   patients.

`RUN_EXTENDED_TUNING=False` is a safety guard: normal execution is cache-only.

<!-- paper-tuning-postrun:begin -->
**Completed cache loaded:** run signature `3fc4b61a32467cfb277ceb1a5a9b670b9620537365426b93af2605e415c69f09`. The notebook remains cache-only (`RUN_EXTENDED_TUNING=False`); the tables below are derived from persisted validation/test artifacts.
<!-- paper-tuning-postrun:end -->


In [5]:
profile_spec = PAPER_PROFILES[PAPER_TUNING_CONFIG.profile]
n_jobs_to_tune = (
    len(profile_spec["horizons"])
    * len(profile_spec["feature_sets"])
    * len(profile_spec["models"])
)
candidate_configurations = (
    n_jobs_to_tune * PAPER_TUNING_CONFIG.n_iter
)
inner_cv_fits = (
    n_jobs_to_tune
    * PAPER_TUNING_CONFIG.n_iter
    * PAPER_TUNING_CONFIG.cv
)

sampling_budget_rows = []
for h in profile_spec["horizons"]:
    _, y_budget = extract_Xy_align(cohorts[h], "CV17", f"y{h}")
    idx_train_budget, _, _ = split_60_20_20(y_budget, seed=SEED)
    n_candidates = len(paper_sampling_candidates(y_budget.iloc[idx_train_budget]))
    sampling_budget_rows.append({
        "horizon": h,
        "sampling_candidates_per_member_set": n_candidates,
        "repeats": PAPER_TUNING_CONFIG.sampling_repeats,
        "feature_sets": len(profile_spec["feature_sets"]),
        "ensemble_members": 3,
        "candidate_fits": (
            n_candidates
            * PAPER_TUNING_CONFIG.sampling_repeats
            * len(profile_spec["feature_sets"])
            * 3
        ),
    })

budget = pd.DataFrame([{
    "profile": PAPER_TUNING_CONFIG.profile,
    "feature_sets": len(profile_spec["feature_sets"]),
    "horizons": len(profile_spec["horizons"]),
    "models": len(profile_spec["models"]),
    "search_jobs": n_jobs_to_tune,
    "random_draws_per_job": PAPER_TUNING_CONFIG.n_iter,
    "candidate_configurations": candidate_configurations,
    "inner_cv_folds": PAPER_TUNING_CONFIG.cv,
    "approx_inner_cv_fits": inner_cv_fits,
    "sampling_refinement": PAPER_TUNING_CONFIG.sampling_refinement,
}])
display(Markdown("### Declared search budget"))
display(budget)
display(pd.DataFrame(sampling_budget_rows))

display(Markdown("### Pingitore search spaces (with documented API fixes)"))
display(paper_search_space_summary())

if RUN_EXTENDED_TUNING:
    display(Markdown(
        "**Explicit opt-in active:** starting/resuming extended tuning. "
        "The test partition remains excluded from every selection step."))
    paper_tuning_run = run_paper_tuning(
        cohorts, config=PAPER_TUNING_CONFIG, force=False)
else:
    paper_tuning_run = load_cached_paper_tuning(
        cohorts, config=PAPER_TUNING_CONFIG)

print("Paper tuning status:", paper_tuning_run.status)
print(paper_tuning_run.message)
if paper_tuning_run.status == "cache_missing":
    display(Markdown(
        "**Extended-tuning cache not found. No search was launched.** "
        "To compute it intentionally, set `RUN_EXTENDED_TUNING=True` and "
        "execute this cell. With the primary profile this means "
        f"**{candidate_configurations:,} candidate configurations** and "
        f"approximately **{inner_cv_fits:,} inner-CV fits**, plus sampling "
        "refinement. Until that cache exists, all pre-existing numbers "
        "remain sensitivity results, not definitive post-tuning results."))


### Declared search budget

,profile,feature_sets,horizons,models,search_jobs,random_draws_per_job,candidate_configurations,inner_cv_folds,approx_inner_cv_fits,sampling_refinement
0,primary,2,2,8,32,5000,160000,2,320000,True


,horizon,sampling_candidates_per_member_set,repeats,feature_sets,ensemble_members,candidate_fits
0,7,63,5,2,3,1890
1,10,9,5,2,3,270


### Pingitore search spaces (with documented API fixes)

,model,branch,parameter,distribution,compatibility_note
0,LogisticRegression,1,clf__C,"randint(args=(1, 10), kwds={})",conditional solver/penalty compatibility
1,LogisticRegression,1,clf__dual,[False],conditional solver/penalty compatibility
2,LogisticRegression,1,clf__max_iter,"randint(args=(50, 500), kwds={})",conditional solver/penalty compatibility
3,LogisticRegression,1,clf__penalty,['l1'],conditional solver/penalty compatibility
4,LogisticRegression,1,clf__solver,['liblinear'],conditional solver/penalty compatibility
...,...,...,...,...,...
80,XGBoost,2,clf__learning_rate,"uniform(args=(0.05, 0.5), kwds={})",canonical names; booster-conditional parameters
81,XGBoost,2,clf__n_estimators,"randint(args=(10, 100), kwds={})",canonical names; booster-conditional parameters
82,XGBoost,2,clf__reg_alpha,"uniform(args=(0.0, 0.5), kwds={})",canonical names; booster-conditional parameters
83,XGBoost,2,clf__reg_lambda,"uniform(args=(0.5, 1.5), kwds={})",canonical names; booster-conditional parameters


Paper tuning status: cache_complete
Complete cached run loaded.


In [6]:
if paper_tuning_run.status in {"computed", "cache_complete", "cache_partial"}:
    display(Markdown("### Individual tuned models — validation and held-out test"))
    display(paper_tuning_run.search_results.round(4))

    if not paper_tuning_run.sampling_results.empty:
        display(Markdown("### Sampling refinement — LR/RF/AdaBoost (validation-selected)"))
        display(paper_tuning_run.sampling_results.round(4))

    display(Markdown("### Tuned paper ensemble — validation and held-out test"))
    display(paper_tuning_run.ensemble_results.round(4))

    display(Markdown("### Paired incremental value on the untouched test"))
    display(paper_tuning_run.incremental_results.round(4))
else:
    display(Markdown(
        "Tuned result tables are intentionally empty until a complete matching "
        "cache is produced."))


### Individual tuned models — validation and held-out test

,horizon,feature_set,model,protocol,threshold,signature,cache_hit,best_cv_score,split,n,f1_macro,roc_auc,precision_1,recall_1,precision_0,recall_0,brier
0,7,CV17,LogisticRegression,paper_random_search,0.5,5576e0526385232150ccf74ed977d697a255026ba12a83...,True,0.7106,validation,878,0.7315,0.8359,0.5088,0.6864,0.9185,0.8420,0.1373
1,7,CV17,LogisticRegression,paper_random_search,0.5,5576e0526385232150ccf74ed977d697a255026ba12a83...,True,0.7106,test,878,0.7284,0.8397,0.4938,0.7143,0.9244,0.8268,0.1375
2,7,CV17,SVC,paper_random_search,0.5,50ef390f2ba9ca5fc4291974fbba86880a41beb54a8176...,True,0.6554,validation,878,0.5153,0.7500,0.5909,0.0769,0.8178,0.9873,0.1347
3,7,CV17,SVC,paper_random_search,0.5,50ef390f2ba9ca5fc4291974fbba86880a41beb54a8176...,True,0.6554,test,878,0.5286,0.7536,0.7143,0.0893,0.8215,0.9915,0.1327
4,7,CV17,KNeighbors,paper_random_search,0.5,014cfa341c4d7033f42309709a04687ac5277f2b880b86...,True,0.6410,validation,878,0.6602,0.7368,0.5978,0.3254,0.8550,0.9478,0.1359
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,10,CV17_THY_CONT_STATES,MLP,paper_random_search,0.5,b4b1a7eb9999cb62ccf0eb8875df1b8f0e3cfc55a9d168...,True,0.7929,test,523,0.7634,0.8460,0.7353,0.6510,0.8102,0.8640,0.1505
60,10,CV17_THY_CONT_STATES,GradientBoosting,paper_random_search,0.5,f85f0f3a8760fbf7a3ebda502942ceb01c8190845ebc3b...,True,0.7945,validation,523,0.7199,0.8077,0.6938,0.5751,0.7741,0.8515,0.1699
61,10,CV17_THY_CONT_STATES,GradientBoosting,paper_random_search,0.5,f85f0f3a8760fbf7a3ebda502942ceb01c8190845ebc3b...,True,0.7945,test,523,0.7646,0.8559,0.7299,0.6615,0.8138,0.8580,0.1489
62,10,CV17_THY_CONT_STATES,XGBoost,paper_random_search,0.5,039d5d96b1d3d082e0951acf6db050f1ba22499855f515...,True,0.7879,validation,523,0.7317,0.8157,0.7055,0.5959,0.7833,0.8545,0.1644


### Sampling refinement — LR/RF/AdaBoost (validation-selected)

,horizon,feature_set,model,protocol,threshold,signature,cache_hit,under_ratio,over_sampler,k_neighbors,selection_validation_f1,split,n,f1_macro,roc_auc,precision_1,recall_1,precision_0,recall_0,brier
0,7,CV17,LogisticRegression,paper_sampling_refinement,0.5,f8aceb7f05c9ab088574a5c50f623906bd56da4a53ea2a...,False,NaN,SVMSMOTE,4,0.7171,validation,878,0.7098,0.8362,0.4648,0.7041,0.9196,0.8068,0.1502
1,7,CV17,LogisticRegression,paper_sampling_refinement,0.5,f8aceb7f05c9ab088574a5c50f623906bd56da4a53ea2a...,False,NaN,SVMSMOTE,4,0.7171,test,878,0.7226,0.8409,0.4755,0.7500,0.9315,0.8042,0.1475
2,7,CV17,RandomForest,paper_sampling_refinement,0.5,aa7f57f276cecdfb6849e91c222d4304623853b8986314...,False,NaN,SVMSMOTE,2,0.7368,validation,878,0.7301,0.8228,0.5655,0.5621,0.8958,0.8970,0.1262
3,7,CV17,RandomForest,paper_sampling_refinement,0.5,aa7f57f276cecdfb6849e91c222d4304623853b8986314...,False,NaN,SVMSMOTE,2,0.7368,test,878,0.7443,0.8308,0.5691,0.6131,0.9067,0.8901,0.1232
4,7,CV17,AdaBoost,paper_sampling_refinement,0.5,5d75a53c7edf1ecaa4d5bb82c4d3b5924e2cd6d451affe...,False,0.25,SMOTE,4,0.7195,validation,878,0.7269,0.8419,0.5167,0.6391,0.9088,0.8575,0.2081
5,7,CV17,AdaBoost,paper_sampling_refinement,0.5,5d75a53c7edf1ecaa4d5bb82c4d3b5924e2cd6d451affe...,False,0.25,SMOTE,4,0.7195,test,878,0.7280,0.8185,0.5142,0.6488,0.9114,0.8549,0.2101
6,7,CV17_THY_CONT_STATES,LogisticRegression,paper_sampling_refinement,0.5,bf8496a7a2e4a73909815d6fafb248d919a578ae7fda93...,False,0.25,SVMSMOTE,4,0.7040,validation,878,0.7002,0.8269,0.4504,0.6982,0.9172,0.7969,0.1550
7,7,CV17_THY_CONT_STATES,LogisticRegression,paper_sampling_refinement,0.5,bf8496a7a2e4a73909815d6fafb248d919a578ae7fda93...,False,0.25,SVMSMOTE,4,0.7040,test,878,0.7207,0.8359,0.4691,0.7679,0.9353,0.7944,0.1517
8,7,CV17_THY_CONT_STATES,RandomForest,paper_sampling_refinement,0.5,44ed56c8486931eda54c908c3234282860db761b04707b...,False,NaN,SVMSMOTE,3,0.7328,validation,878,0.7338,0.8233,0.5714,0.5680,0.8972,0.8984,0.1228
9,7,CV17_THY_CONT_STATES,RandomForest,paper_sampling_refinement,0.5,44ed56c8486931eda54c908c3234282860db761b04707b...,False,NaN,SVMSMOTE,3,0.7328,test,878,0.7350,0.8418,0.5673,0.5774,0.8996,0.8958,0.1177


### Tuned paper ensemble — validation and held-out test

,horizon,feature_set,model,protocol,threshold,signature,cache_hit,split,n,f1_macro,roc_auc,precision_1,recall_1,precision_0,recall_0,brier
0,7,CV17,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,26789f1405052da3832dcc652528101af6179cef3c5592...,False,validation,878,0.7235,0.8366,0.5094,0.6391,0.9084,0.8533,0.1426
1,7,CV17,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,26789f1405052da3832dcc652528101af6179cef3c5592...,False,test,878,0.7386,0.8411,0.5227,0.6845,0.9195,0.8521,0.1414
2,7,CV17_THY_CONT_STATES,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,e1ce32077cafe1261e41d2d96d1a46c2d082739b062b65...,False,validation,878,0.7199,0.8333,0.5000,0.6450,0.9091,0.8463,0.1409
3,7,CV17_THY_CONT_STATES,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,e1ce32077cafe1261e41d2d96d1a46c2d082739b062b65...,False,test,878,0.7510,0.8432,0.5328,0.7262,0.9291,0.8493,0.1385
4,10,CV17,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,9640ce12a862e1e703b3b65607817fd63797398ef8cea8...,False,validation,523,0.7381,0.8107,0.6569,0.6943,0.8150,0.7879,0.1716
5,10,CV17,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,9640ce12a862e1e703b3b65607817fd63797398ef8cea8...,False,test,523,0.7616,0.8555,0.6473,0.8125,0.8723,0.7432,0.1593
6,10,CV17_THY_CONT_STATES,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,3f8deb569ca22efb4bba3d8c2bcd7637aaa3a4a6997177...,False,validation,523,0.7440,0.8142,0.6650,0.6995,0.8188,0.7939,0.1715
7,10,CV17_THY_CONT_STATES,ENSEMBLE_PAPER,paper_ensemble_sampling_refined,0.5,3f8deb569ca22efb4bba3d8c2bcd7637aaa3a4a6997177...,False,test,523,0.7570,0.8558,0.6456,0.7969,0.8636,0.7462,0.1599


### Paired incremental value on the untouched test

,horizon,base,feature_set,model,split,threshold,delta_f1_macro,delta_f1_ci_lo,delta_f1_ci_hi,delta_auroc,delta_auroc_ci_lo,delta_auroc_ci_hi
0,7,CV17,CV17_THY_CONT_STATES,ENSEMBLE_PAPER,test,0.5,0.0124,-0.0051,0.0322,0.0021,-0.0067,0.0105
1,10,CV17,CV17_THY_CONT_STATES,ENSEMBLE_PAPER,test,0.5,-0.0046,-0.0227,0.0120,0.0003,-0.0065,0.0069


## Sensitivity S1 — fixed-parameter classification and ensembles

Questa sezione conserva integralmente l'analisi già calcolata con parametri
prespecificati. Va interpretata come **sensibilità pre-tuning**, non come il
risultato primario del nuovo flusso Pingitore-like.

All models and ensembles use the leakage-free pipeline
`SimpleImputer → StandardScaler → RandomOverSampler → estimator`. Decision
thresholds maximise F1-macro on inner-validation (A) or validation (B).
Reported for both schemes, horizons and all feature sets: F1-macro, AUROC,
precision/recall and Brier.


In [7]:
clf = RA.build_classification_table(cohorts, horizons=HORIZONS, seed=SEED, force=FORCE)
print("classification rows:", len(clf))

def clf_view(scheme, horizon, metrics=("f1_macro", "roc_auc", "precision_1",
                                       "recall_1", "brier")):
    sub = clf[(clf["scheme"] == scheme) & (clf["horizon"] == horizon)].copy()
    piv = sub.pivot_table(index="feature_set", columns="model",
                          values="f1_macro")
    return sub, piv

for h in HORIZONS:
    for scheme, label in [("A_5fold_cv", "A · 5-fold CV"),
                          ("B_60_20_20", "B · 60/20/20 test")]:
        sub = clf[(clf["scheme"] == scheme) & (clf["horizon"] == h)]
        display(Markdown(f"### Classification — strict cohort, {h}y, scheme {label}"))
        cols = ["feature_set", "model", "f1_macro", "roc_auc",
                "precision_1", "recall_1", "precision_0", "recall_0", "brier"]
        display(sub[cols].round(3).reset_index(drop=True))


classification rows:

 480


### Classification — strict cohort, 7y, scheme A · 5-fold CV

,feature_set,model,f1_macro,roc_auc,precision_1,recall_1,precision_0,recall_0,brier
0,CV17,LogisticRegression,0.731,0.840,0.541,0.612,0.905,0.874,0.162
1,CV17,RandomForest,0.707,0.816,0.500,0.594,0.899,0.856,0.123
2,CV17,AdaBoost,0.720,0.833,0.536,0.572,0.897,0.882,0.196
3,CV17,SVC,0.715,0.816,0.517,0.581,0.897,0.869,0.153
4,CV17,KNeighbors,0.640,0.710,0.395,0.491,0.871,0.818,0.216
...,...,...,...,...,...,...,...,...,...
115,CV17_THY_CONT_RATIO,ENS_VOTE_BOOST,0.724,0.836,0.534,0.587,0.900,0.878,0.145
116,CV17_THY_CONT_RATIO,ENS_VOTE_DIVERSE,0.734,0.841,0.551,0.609,0.904,0.880,0.129
117,CV17_THY_CONT_RATIO,ENS_VOTE_WIDE,0.727,0.839,0.538,0.607,0.904,0.873,0.124
118,CV17_THY_CONT_RATIO,ENS_STACK,0.734,0.845,0.554,0.604,0.904,0.882,0.109


### Classification — strict cohort, 7y, scheme B · 60/20/20 test

,feature_set,model,f1_macro,roc_auc,precision_1,recall_1,precision_0,recall_0,brier
0,CV17,LogisticRegression,0.744,0.838,0.617,0.548,0.896,0.920,0.161
1,CV17,RandomForest,0.718,0.830,0.553,0.530,0.890,0.899,0.118
2,CV17,AdaBoost,0.715,0.824,0.593,0.476,0.882,0.923,0.197
3,CV17,SVC,0.726,0.826,0.520,0.625,0.907,0.863,0.147
4,CV17,KNeighbors,0.671,0.745,0.444,0.518,0.881,0.846,0.199
...,...,...,...,...,...,...,...,...,...
115,CV17_THY_CONT_RATIO,ENS_VOTE_BOOST,0.748,0.837,0.579,0.613,0.907,0.894,0.140
116,CV17_THY_CONT_RATIO,ENS_VOTE_DIVERSE,0.752,0.848,0.568,0.649,0.914,0.883,0.124
117,CV17_THY_CONT_RATIO,ENS_VOTE_WIDE,0.756,0.848,0.572,0.661,0.917,0.883,0.121
118,CV17_THY_CONT_RATIO,ENS_STACK,0.753,0.849,0.662,0.536,0.895,0.935,0.105


### Classification — strict cohort, 10y, scheme A · 5-fold CV

,feature_set,model,f1_macro,roc_auc,precision_1,recall_1,precision_0,recall_0,brier
0,CV17,LogisticRegression,0.761,0.852,0.721,0.682,0.821,0.838,0.156
1,CV17,RandomForest,0.751,0.836,0.690,0.697,0.824,0.809,0.157
2,CV17,AdaBoost,0.753,0.837,0.689,0.686,0.818,0.819,0.193
3,CV17,SVC,0.754,0.840,0.689,0.689,0.819,0.818,0.160
4,CV17,KNeighbors,0.697,0.764,0.601,0.662,0.791,0.743,0.203
...,...,...,...,...,...,...,...,...,...
115,CV17_THY_CONT_RATIO,ENS_VOTE_BOOST,0.755,0.849,0.697,0.701,0.827,0.814,0.156
116,CV17_THY_CONT_RATIO,ENS_VOTE_DIVERSE,0.767,0.855,0.700,0.724,0.837,0.815,0.149
117,CV17_THY_CONT_RATIO,ENS_VOTE_WIDE,0.772,0.855,0.709,0.722,0.837,0.825,0.150
118,CV17_THY_CONT_RATIO,ENS_STACK,0.767,0.856,0.707,0.706,0.829,0.829,0.146


### Classification — strict cohort, 10y, scheme B · 60/20/20 test

,feature_set,model,f1_macro,roc_auc,precision_1,recall_1,precision_0,recall_0,brier
0,CV17,LogisticRegression,0.763,0.853,0.670,0.760,0.849,0.782,0.164
1,CV17,RandomForest,0.742,0.835,0.634,0.766,0.845,0.743,0.159
2,CV17,AdaBoost,0.737,0.835,0.642,0.719,0.825,0.767,0.190
3,CV17,SVC,0.762,0.855,0.665,0.766,0.851,0.776,0.157
4,CV17,KNeighbors,0.702,0.788,0.705,0.510,0.755,0.876,0.193
...,...,...,...,...,...,...,...,...,...
115,CV17_THY_CONT_RATIO,ENS_VOTE_BOOST,0.744,0.850,0.645,0.740,0.835,0.764,0.157
116,CV17_THY_CONT_RATIO,ENS_VOTE_DIVERSE,0.753,0.859,0.668,0.724,0.832,0.792,0.150
117,CV17_THY_CONT_RATIO,ENS_VOTE_WIDE,0.745,0.859,0.632,0.786,0.856,0.734,0.151
118,CV17_THY_CONT_RATIO,ENS_STACK,0.743,0.855,0.656,0.714,0.825,0.782,0.150


In [8]:
# Focus: ENSEMBLE across feature sets, both schemes, both horizons.
ens = clf[clf["model"] == "ENSEMBLE"].copy()
display(Markdown("### ENSEMBLE — F1-macro / AUROC by feature set, scheme, horizon"))
ens_tab = ens.pivot_table(index="feature_set",
                          columns=["horizon", "scheme"],
                          values=["f1_macro", "roc_auc"]).round(3)
display(ens_tab)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, h in zip(axes, HORIZONS):
    for scheme, mk in [("A_5fold_cv", "o-"), ("B_60_20_20", "s--")]:
        d = ens[(ens["horizon"] == h) & (ens["scheme"] == scheme)]
        d = d.set_index("feature_set").reindex(
            describe_feature_sets()["feature_set"])
        ax.plot(d.index, d["f1_macro"], mk, label=f"{scheme}")
    ax.set_title(f"ENSEMBLE F1-macro — strict {h}y", fontweight="bold")
    ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3); ax.legend()
axes[0].set_ylabel("F1-macro")
fig.tight_layout(); plt.show()


### ENSEMBLE — F1-macro / AUROC by feature set, scheme, horizon

f1_macro                                     roc_auc                                 
horizon                 7                     10                    7                     10           
scheme          A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20
feature_set                                                                                            
CV17                 0.731      0.748      0.768      0.753      0.842      0.843      0.854      0.850
CV17_THY_ABNORMAL_BIN             0.722      0.718      0.763      0.763      0.842      0.844      0.855      0.853
CV17_THY_STATES             0.724      0.748      0.768      0.745      0.842      0.843      0.855      0.853
CV17_THY_CONT            0.736      0.744      0.772      0.748      0.845      0.845      0.857      0.853
CV17_THY_STATE_ORD             0.725      0.763      0.773      0.758      0.843      0.842      0.855      0.851
CV17_THY_CONT_STATES_RATIO           0.727      0.717      0.773      0.737      0.845      0.846      0.857      0.853
CV17_THY_CONT_RATIO      0.729      0.762      0.768      0.746      0.846      0.846      0.857      0.854
CV17_THY_CONT_STATES           0.729      0.736      0.768      0.741      0.845      0.845      0.855      0.854

## Sensitivity S1b — pre-tuning ensemble bake-off

Analisi esplorativa con parametri fissi: il paper `ENSEMBLE` (soft vote
LR+RF+AdaBoost) è confrontato con combinazioni alternative.

| name | method | base members |
|---|---|---|
| `ENSEMBLE` *(paper)* | soft vote | LR, RF, AdaBoost |
| `ENS_VOTE_XGB` | soft vote | LR, RF, XGBoost |
| `ENS_VOTE_BOOST` | soft vote | AdaBoost, GB, XGBoost |
| `ENS_VOTE_DIVERSE` | soft vote | LR, RF, XGB, SVC |
| `ENS_VOTE_WIDE` | soft vote | LR, RF, XGB, MLP |
| `ENS_STACK` | stacking (meta-LR) | LR, RF, XGB |
| `ENS_STACK_GB` | stacking (meta-LR) | LR, RF, GB, XGB |

La selezione usa soltanto lo schema A su CV17, mediando 7 e 10 anni. Il test B
non partecipa alla scelta. `WINNER` resta un risultato di sensibilità e non
sostituisce l'ensemble paper-like ottenuto dopo tuning esteso.


In [9]:
bo = clf[clf["model"].isin(ENSEMBLE_NAMES)].copy()

# Exploratory ranking on CV17 using cross-validation only (never test B).
selection = bo[(bo["feature_set"] == "CV17") &
               (bo["scheme"] == "A_5fold_cv")]
rank = (selection
        .groupby("model")[["f1_macro", "roc_auc", "brier"]].mean()
        .sort_values(["f1_macro", "roc_auc"], ascending=False).round(4))
display(Markdown("### Exploratory ensemble ranking on CV17 (A only; mean 7/10y)"))
display(rank)

# Per-cell F1-macro on CV17, ensembles x (horizon, scheme).
cv17 = bo[bo["feature_set"] == "CV17"]
cell = cv17.pivot_table(index="model", columns=["horizon", "scheme"],
                        values="f1_macro").round(3)
cell = cell.reindex(rank.index)
display(Markdown("### F1-macro per cell on CV17 (ensembles × horizon × scheme)"))
display(cell)

# Delta F1-macro vs paper ENSEMBLE, same cells.
paper = cv17[cv17["model"] == "ENSEMBLE"].set_index(["horizon", "scheme"])["f1_macro"]
delta = cv17.copy()
delta["delta_vs_paper"] = delta.apply(
    lambda r: r["f1_macro"] - paper.loc[(r["horizon"], r["scheme"])], axis=1)
dpiv = delta.pivot_table(index="model", columns=["horizon", "scheme"],
                         values="delta_vs_paper").round(3).reindex(rank.index)
display(Markdown("### Δ F1-macro vs paper ENSEMBLE (CV17; >0 = better than paper)"))
display(dpiv)

WINNER = RA.pick_winner(clf)
print("WINNER (best non-paper ensemble on CV17):", WINNER)


### Exploratory ensemble ranking on CV17 (A only; mean 7/10y)

,f1_macro,roc_auc,brier
model,,,
ENSEMBLE,0.7492,0.8477,0.1492
ENS_STACK,0.7481,0.8478,0.1291
ENS_STACK_GB,0.7446,0.8479,0.1291
ENS_VOTE_DIVERSE,0.7421,0.8444,0.1427
ENS_VOTE_WIDE,0.7414,0.8440,0.1414
ENS_VOTE_XGB,0.7396,0.8438,0.1415
ENS_VOTE_BOOST,0.7367,0.8397,0.1541


### F1-macro per cell on CV17 (ensembles × horizon × scheme)

horizon                  7                     10           
scheme           A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20
model                                                       
ENSEMBLE              0.731      0.748      0.768      0.753
ENS_STACK             0.726      0.756      0.770      0.750
ENS_STACK_GB          0.727      0.762      0.762      0.748
ENS_VOTE_DIVERSE      0.718      0.759      0.766      0.757
ENS_VOTE_WIDE         0.719      0.753      0.763      0.749
ENS_VOTE_XGB          0.721      0.754      0.758      0.750
ENS_VOTE_BOOST        0.713      0.753      0.760      0.757

### Δ F1-macro vs paper ENSEMBLE (CV17; >0 = better than paper)

horizon                  7                     10           
scheme           A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20
model                                                       
ENSEMBLE              0.000      0.000      0.000      0.000
ENS_STACK            -0.004      0.008      0.002     -0.004
ENS_STACK_GB         -0.004      0.015     -0.005     -0.006
ENS_VOTE_DIVERSE     -0.012      0.011     -0.002      0.004
ENS_VOTE_WIDE        -0.011      0.005     -0.004     -0.004
ENS_VOTE_XGB         -0.009      0.006     -0.010     -0.004
ENS_VOTE_BOOST       -0.017      0.005     -0.008      0.003

WINNER (best non-paper ensemble on CV17): ENS_STACK


In [10]:
# Visual: F1-macro by feature set for every ensemble (strict 7y, scheme A).
fsorder = describe_feature_sets()["feature_set"].tolist()
fig, ax = plt.subplots(figsize=(13, 6))
for m in ENSEMBLE_NAMES:
    d = bo[(bo["model"] == m) & (bo["horizon"] == 7) &
           (bo["scheme"] == "A_5fold_cv")]
    d = d.set_index("feature_set").reindex(fsorder)
    style = "o-" if m != WINNER else "D-"
    lw = 1.5 if m not in ("ENSEMBLE", WINNER) else 2.8
    ax.plot(d.index, d["f1_macro"], style, linewidth=lw, label=m)
ax.set_title("Ensembles — F1-macro by feature set (strict 7y, scheme A)",
             fontweight="bold")
ax.set_ylabel("F1-macro"); ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
fig.tight_layout(); plt.show()


## Sensitivity S2 — incremental value before extended tuning

Analisi appaiata già esistente, mantenuta come sensibilità: ogni feature set
tiroideo vs CV17 sul medesimo ensemble a parametri fissi, con Δ F1-macro e
Δ AUROC (bootstrap 95%). Le soglie hard sono selezionate esclusivamente su
inner-validation/validation e mai sul test/OOF. I confronti secondari non sono
corretti per molteplicità.


In [11]:
incr = RA.build_incremental_table(cohorts, horizons=HORIZONS,
                                  n_boot=N_BOOT, seed=SEED, force=FORCE)

def fmt_ci(row, point_col, ci_prefix):
    return (f"{row[point_col]:+.3f} "
            f"[{row[ci_prefix + '_ci_lo']:+.3f}, "
            f"{row[ci_prefix + '_ci_hi']:+.3f}]")

incr_disp = incr.copy()
incr_disp["ΔF1-macro [CI95]"] = incr_disp.apply(
    lambda r: fmt_ci(r, "delta_f1_macro", "delta_f1"), axis=1)
incr_disp["ΔAUROC [CI95]"]   = incr_disp.apply(
    lambda r: fmt_ci(r, "delta_auroc", "delta_auroc"), axis=1)
for h in HORIZONS:
    display(Markdown(f"### Incremental value vs CV17 — strict {h}y (ENSEMBLE)"))
    sub = incr_disp[incr_disp["horizon"] == h]
    display(sub[["feature_set", "scheme", "ΔF1-macro [CI95]",
                 "ΔAUROC [CI95]"]].reset_index(drop=True))


### Incremental value vs CV17 — strict 7y (ENSEMBLE)

,feature_set,scheme,ΔF1-macro [CI95],ΔAUROC [CI95]
0,CV17_THY_CONT_STATES,A_5fold_cv,"-0.001 [-0.011, +0.009]","+0.003 [-0.000, +0.007]"
1,CV17_THY_CONT_STATES,B_60_20_20,"-0.012 [-0.040, +0.019]","+0.002 [-0.007, +0.011]"
2,CV17_THY_ABNORMAL_BIN,A_5fold_cv,"-0.008 [-0.017, +0.001]","+0.000 [-0.002, +0.003]"
3,CV17_THY_ABNORMAL_BIN,B_60_20_20,"-0.030 [-0.064, +0.008]","+0.001 [-0.006, +0.008]"
4,CV17_THY_STATE_ORD,A_5fold_cv,"-0.005 [-0.013, +0.001]","+0.002 [+0.000, +0.004]"
5,CV17_THY_STATE_ORD,B_60_20_20,"+0.015 [-0.009, +0.038]","-0.001 [-0.005, +0.004]"
6,CV17_THY_CONT,A_5fold_cv,"+0.006 [-0.003, +0.015]","+0.004 [+0.001, +0.008]"
7,CV17_THY_CONT,B_60_20_20,"-0.004 [-0.034, +0.030]","+0.002 [-0.005, +0.010]"
8,CV17_THY_STATES,A_5fold_cv,"-0.006 [-0.014, +0.003]","+0.000 [-0.002, +0.003]"
9,CV17_THY_STATES,B_60_20_20,"-0.000 [-0.025, +0.025]","-0.000 [-0.006, +0.006]"


### Incremental value vs CV17 — strict 10y (ENSEMBLE)

,feature_set,scheme,ΔF1-macro [CI95],ΔAUROC [CI95]
0,CV17_THY_CONT_STATES,A_5fold_cv,"+0.001 [-0.009, +0.011]","+0.002 [-0.002, +0.006]"
1,CV17_THY_CONT_STATES,B_60_20_20,"-0.013 [-0.037, +0.009]","+0.003 [-0.004, +0.011]"
2,CV17_THY_ABNORMAL_BIN,A_5fold_cv,"-0.005 [-0.012, +0.003]","+0.001 [-0.001, +0.003]"
3,CV17_THY_ABNORMAL_BIN,B_60_20_20,"+0.010 [-0.007, +0.026]","+0.002 [-0.003, +0.008]"
4,CV17_THY_STATE_ORD,A_5fold_cv,"+0.005 [-0.002, +0.013]","+0.001 [-0.001, +0.003]"
5,CV17_THY_STATE_ORD,B_60_20_20,"+0.005 [-0.011, +0.021]","+0.000 [-0.004, +0.005]"
6,CV17_THY_CONT,A_5fold_cv,"+0.004 [-0.005, +0.013]","+0.003 [-0.000, +0.006]"
7,CV17_THY_CONT,B_60_20_20,"-0.005 [-0.033, +0.020]","+0.003 [-0.004, +0.009]"
8,CV17_THY_STATES,A_5fold_cv,"+0.000 [-0.007, +0.007]","+0.002 [-0.000, +0.004]"
9,CV17_THY_STATES,B_60_20_20,"-0.009 [-0.032, +0.015]","+0.002 [-0.002, +0.007]"


In [12]:
# Same incremental analysis for the WINNER ensemble (cached, suffixed).
incr_w = RA.build_incremental_table(cohorts, horizons=HORIZONS, model_name=WINNER,
                                    n_boot=N_BOOT, seed=SEED, force=FORCE,
                                    out_suffix=WINNER)
incr_w_disp = incr_w.copy()
incr_w_disp["ΔF1-macro [CI95]"] = incr_w_disp.apply(
    lambda r: fmt_ci(r, "delta_f1_macro", "delta_f1"), axis=1)
incr_w_disp["ΔAUROC [CI95]"]   = incr_w_disp.apply(
    lambda r: fmt_ci(r, "delta_auroc", "delta_auroc"), axis=1)
for h in HORIZONS:
    display(Markdown(f"### Incremental value vs CV17 — strict {h}y (WINNER: {WINNER})"))
    sub = incr_w_disp[incr_w_disp["horizon"] == h]
    display(sub[["feature_set", "scheme", "ΔF1-macro [CI95]",
                 "ΔAUROC [CI95]"]].reset_index(drop=True))


### Incremental value vs CV17 — strict 7y (WINNER: ENS_STACK)

,feature_set,scheme,ΔF1-macro [CI95],ΔAUROC [CI95]
0,CV17_THY_CONT_STATES,A_5fold_cv,"+0.003 [-0.007, +0.014]","+0.002 [-0.002, +0.005]"
1,CV17_THY_CONT_STATES,B_60_20_20,"-0.016 [-0.042, +0.011]","+0.001 [-0.006, +0.009]"
2,CV17_THY_ABNORMAL_BIN,A_5fold_cv,"+0.001 [-0.007, +0.010]","+0.000 [-0.002, +0.002]"
3,CV17_THY_ABNORMAL_BIN,B_60_20_20,"-0.015 [-0.039, +0.010]","-0.001 [-0.006, +0.004]"
4,CV17_THY_STATE_ORD,A_5fold_cv,"-0.002 [-0.010, +0.005]","+0.001 [-0.000, +0.003]"
5,CV17_THY_STATE_ORD,B_60_20_20,"+0.007 [-0.012, +0.029]","-0.002 [-0.005, +0.001]"
6,CV17_THY_CONT,A_5fold_cv,"+0.009 [+0.000, +0.019]","+0.003 [+0.000, +0.006]"
7,CV17_THY_CONT,B_60_20_20,"-0.002 [-0.023, +0.020]","+0.001 [-0.005, +0.008]"
8,CV17_THY_STATES,A_5fold_cv,"+0.002 [-0.005, +0.010]","-0.001 [-0.003, +0.001]"
9,CV17_THY_STATES,B_60_20_20,"+0.001 [-0.016, +0.019]","-0.001 [-0.007, +0.005]"


### Incremental value vs CV17 — strict 10y (WINNER: ENS_STACK)

,feature_set,scheme,ΔF1-macro [CI95],ΔAUROC [CI95]
0,CV17_THY_CONT_STATES,A_5fold_cv,"+0.001 [-0.009, +0.011]","+0.001 [-0.003, +0.004]"
1,CV17_THY_CONT_STATES,B_60_20_20,"-0.003 [-0.023, +0.019]","+0.002 [-0.007, +0.010]"
2,CV17_THY_ABNORMAL_BIN,A_5fold_cv,"-0.002 [-0.011, +0.007]","-0.000 [-0.002, +0.002]"
3,CV17_THY_ABNORMAL_BIN,B_60_20_20,"+0.003 [-0.014, +0.020]","+0.003 [-0.002, +0.008]"
4,CV17_THY_STATE_ORD,A_5fold_cv,"+0.001 [-0.007, +0.009]","+0.001 [-0.001, +0.002]"
5,CV17_THY_STATE_ORD,B_60_20_20,"+0.001 [-0.010, +0.013]","+0.000 [-0.004, +0.005]"
6,CV17_THY_CONT,A_5fold_cv,"+0.005 [-0.005, +0.015]","+0.002 [-0.001, +0.006]"
7,CV17_THY_CONT,B_60_20_20,"-0.008 [-0.030, +0.014]","+0.002 [-0.006, +0.011]"
8,CV17_THY_STATES,A_5fold_cv,"+0.001 [-0.008, +0.010]","+0.001 [-0.001, +0.003]"
9,CV17_THY_STATES,B_60_20_20,"+0.011 [-0.005, +0.030]","+0.001 [-0.004, +0.006]"


## Sensitivity S3 — fixed-parameter ML indicator in survival

The pre-tuning ensemble probability is used as an ML composite indicator:
- `p_event` in a single-covariate Cox model → C-index;
- `p_survive = 1 − p_event` in KM stratification at 0.6 and the median.

CV17 and CV17_THY_CONT_STATES are compared with paired bootstrap Δ C-index. Log-rank
tests describe internal separation and are not incremental tests.

> **Conditional estimand.** This analysis uses the strict cohort after removing
> patients censored before the horizon. It is comparable with the paper but is
> different from the full-cohort competing-risk analysis elsewhere in the
> thesis.


In [13]:
cindex_df, km_df = RA.build_ml_indicator(cohorts, base="CV17",
                                         thy="CV17_THY_CONT_STATES",
                                         horizons=HORIZONS, seed=SEED,
                                         force=FORCE)
display(Markdown("### Cox single-covariate C-index (ML indicator = p_event)"))
display(cindex_df.drop(columns=["cache_key"], errors="ignore").round(4))
display(Markdown("### KM stratification by predicted survival (0.6 and median)"))
display(km_df.drop(columns=["cache_key"], errors="ignore").round(4))


### Cox single-covariate C-index (ML indicator = p_event)

,model,horizon,scheme,feature_set,c_index,n,events,c_index_ci_lo,c_index_ci_hi
0,ENSEMBLE,7,A,CV17,0.8107,4390.0,843.0,NaN,NaN
1,ENSEMBLE,7,A,CV17_THY_CONT_STATES,0.8146,4390.0,843.0,NaN,NaN
2,ENSEMBLE,7,A,DELTA(CV17_THY_CONT_STATES-CV17),0.0039,NaN,NaN,0.0005,0.0073
3,ENSEMBLE,7,B,CV17,0.8204,878.0,168.0,NaN,NaN
4,ENSEMBLE,7,B,CV17_THY_CONT_STATES,0.8236,878.0,168.0,NaN,NaN
5,ENSEMBLE,7,B,DELTA(CV17_THY_CONT_STATES-CV17),0.0032,NaN,NaN,-0.0044,0.0108
6,ENSEMBLE,10,A,CV17,0.7924,2614.0,962.0,NaN,NaN
7,ENSEMBLE,10,A,CV17_THY_CONT_STATES,0.7948,2614.0,962.0,NaN,NaN
8,ENSEMBLE,10,A,DELTA(CV17_THY_CONT_STATES-CV17),0.0024,NaN,NaN,-0.0006,0.0056
9,ENSEMBLE,10,B,CV17,0.7890,523.0,192.0,NaN,NaN


### KM stratification by predicted survival (0.6 and median)

,model,horizon,scheme,feature_set,threshold_name,threshold,n_high,n_low,surv_high,surv_low,logrank_p
0,ENSEMBLE,7,A,CV17,fixed_0.6,0.6000,2818,1572,0.9368,0.5770,0.0
1,ENSEMBLE,7,A,CV17,median,0.6878,2195,2195,0.9572,0.6588,0.0
2,ENSEMBLE,7,A,CV17_THY_CONT_STATES,fixed_0.6,0.6000,2781,1609,0.9407,0.5786,0.0
3,ENSEMBLE,7,A,CV17_THY_CONT_STATES,median,0.6874,2195,2195,0.9540,0.6620,0.0
4,ENSEMBLE,7,B,CV17,fixed_0.6,0.6000,551,327,0.9401,0.5872,0.0
5,ENSEMBLE,7,B,CV17,median,0.6860,439,439,0.9544,0.6629,0.0
6,ENSEMBLE,7,B,CV17_THY_CONT_STATES,fixed_0.6,0.6000,557,321,0.9408,0.5794,0.0
7,ENSEMBLE,7,B,CV17_THY_CONT_STATES,median,0.6888,439,439,0.9522,0.6651,0.0
8,ENSEMBLE,10,A,CV17,fixed_0.6,0.6000,1287,1327,0.8834,0.3881,0.0
9,ENSEMBLE,10,A,CV17,median,0.5946,1307,1307,0.8806,0.3833,0.0


In [14]:
# KM curves for CV17 vs CV17_THY_CONT_STATES at the 0.6 threshold (scheme A, 7y).
from lifelines import KaplanMeierFitter
from src.alignment.ml_indicator import predicted_risk, _survival_frame

H = 7
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, fs in zip(axes, ["CV17", "CV17_THY_CONT_STATES"]):
    risk = predicted_risk(cohorts[H], fs, f"y{H}", scheme="A", seed=SEED)
    surv = _survival_frame(cohorts[H], H)
    data = surv[["surv_time", "surv_event"]].join(risk["p_survive"],
                                                  how="inner").dropna()
    high = data["p_survive"] >= 0.6
    kmf = KaplanMeierFitter()
    for label, mask in [("predicted survivor (p≥0.6)", high),
                        ("predicted death (p<0.6)", ~high)]:
        if mask.sum():
            kmf.fit(data.loc[mask, "surv_time"], data.loc[mask, "surv_event"],
                    label=f"{label}  (n={int(mask.sum())})")
            kmf.plot_survival_function(ax=ax, ci_show=False)
    ax.set_title(f"{fs} — KM by ML indicator (strict {H}y, scheme A)",
                 fontweight="bold")
    ax.set_xlabel("Years"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Survival probability")
fig.tight_layout(); plt.show()


In [15]:
# ML indicator C-index for the WINNER ensemble, side-by-side with the paper one.
cindex_w, km_w = RA.build_ml_indicator(cohorts, base="CV17", thy="CV17_THY_CONT_STATES",
                                       horizons=HORIZONS, model_name=WINNER,
                                       seed=SEED, force=FORCE, out_suffix=WINNER)
display(Markdown(f"### Cox C-index — WINNER ({WINNER}) vs paper ENSEMBLE"))
cmp = []
for h in HORIZONS:
    for scheme in ["A", "B"]:
        def _delta(df):
            r = df[(df["horizon"] == h) & (df["scheme"] == scheme) &
                   (df["feature_set"].str.startswith("DELTA"))]
            return float(r["c_index"].values[0]) if len(r) else float("nan")
        cmp.append({"horizon": h, "scheme": scheme,
                    "ΔC-index paper": round(_delta(cindex_df), 4),
                    f"ΔC-index {WINNER}": round(_delta(cindex_w), 4)})
display(pd.DataFrame(cmp))
display(Markdown(f"### KM stratification — WINNER ({WINNER})"))
display(km_w.drop(columns=["cache_key"], errors="ignore").round(4))


### Cox C-index — WINNER (ENS_STACK) vs paper ENSEMBLE

,horizon,scheme,ΔC-index paper,ΔC-index ENS_STACK
0,7,A,0.0039,0.0022
1,7,B,0.0032,0.0025
2,10,A,0.0024,0.0014
3,10,B,0.0038,0.0033


### KM stratification — WINNER (ENS_STACK)

,model,horizon,scheme,feature_set,threshold_name,threshold,n_high,n_low,surv_high,surv_low,logrank_p
0,ENS_STACK,7,A,CV17,fixed_0.6,0.6000,3630,760,0.8904,0.4145,0.0
1,ENS_STACK,7,A,CV17,median,0.9056,2195,2195,0.9572,0.6588,0.0
2,ENS_STACK,7,A,CV17_THY_CONT_STATES,fixed_0.6,0.6000,3642,748,0.8929,0.3944,0.0
3,ENS_STACK,7,A,CV17_THY_CONT_STATES,median,0.9070,2195,2195,0.9522,0.6638,0.0
4,ENS_STACK,7,B,CV17,fixed_0.6,0.6000,733,145,0.8990,0.3517,0.0
5,ENS_STACK,7,B,CV17,median,0.9044,439,439,0.9544,0.6629,0.0
6,ENS_STACK,7,B,CV17_THY_CONT_STATES,fixed_0.6,0.6000,722,156,0.9017,0.3782,0.0
7,ENS_STACK,7,B,CV17_THY_CONT_STATES,median,0.9058,439,439,0.9522,0.6651,0.0
8,ENS_STACK,10,A,CV17,fixed_0.6,0.6000,1557,1057,0.8439,0.3198,0.0
9,ENS_STACK,10,A,CV17,median,0.7407,1307,1307,0.8806,0.3833,0.0


## Sensitivity S4 — fixed-parameter ablation

Pre-tuning knock-out analysis: replace a feature with its training mean,
recompute F1′ and report importance = F1/F1′. Single-variable and clustered
multi-variable analyses are retained to show where thyroid features sit among
the predictors, but they refer to the fixed-parameter ensemble.


In [16]:
abl = RA.build_ablation(cohorts, horizons=HORIZONS, seed=SEED, force=FORCE)
print("ablation rows:", len(abl))

# Single-variable ranking for the richest thyroid set, both horizons.
for h in HORIZONS:
    sub = abl[(abl["horizon"] == h) & (abl["feature_set"] == "CV17_THY_CONT_STATES")
              & (abl["mode"] == "single")].copy()
    sub = sub.sort_values("importance", ascending=False)
    sub["rank"] = range(1, len(sub) + 1)
    display(Markdown(f"### Ablation (single) — CV17_THY_CONT_STATES, strict {h}y "
                     "— thyroid rows highlighted"))
    display(sub[["rank", "feature", "importance", "is_thyroid"]]
            .round(4).reset_index(drop=True))


ablation rows: 446


### Ablation (single) — CV17_THY_CONT_STATES, strict 7y — thyroid rows highlighted

,rank,feature,importance,is_thyroid
0,1,Age,1.1265,False
1,2,fe,1.0594,False
2,3,Dyslipidemia,1.0282,False
3,4,TSH,1.0194,True
4,5,Angiography,1.0155,False
5,6,Hypertension,1.0150,False
6,7,PostIsch_DCM,1.0134,False
7,8,Gender,1.0110,False
8,9,Acute_MI,1.0092,False
9,10,AFib,1.0090,False


### Ablation (single) — CV17_THY_CONT_STATES, strict 10y — thyroid rows highlighted

,rank,feature,importance,is_thyroid
0,1,Age,1.0461,False
1,2,Diabetes,1.0062,False
2,3,Previous_PCI,1.0030,False
3,4,Angiography,1.0024,False
4,5,PostIsch_DCM,1.0024,False
5,6,Hypothyroid,1.0024,True
6,7,AFib,1.0019,False
7,8,Gender,1.0018,False
8,9,Hypertension,1.0000,False
9,10,SCH,1.0000,True


In [17]:
# Multi-variable (clustered) ablation for CV17_THY_CONT_STATES.
for h in HORIZONS:
    sub = abl[(abl["horizon"] == h) & (abl["feature_set"] == "CV17_THY_CONT_STATES")
              & (abl["mode"] == "multi")].copy()
    sub = sub.sort_values("importance", ascending=False)
    display(Markdown(f"### Ablation (multi-variable, clustered) — CV17_THY_CONT_STATES, strict {h}y"))
    display(sub[["feature", "members", "importance", "is_thyroid"]]
            .round(4).reset_index(drop=True))


### Ablation (multi-variable, clustered) — CV17_THY_CONT_STATES, strict 7y

,feature,members,importance,is_thyroid
0,cluster_5,"Gender, Angina, Previous_CABG, Previous_PCI, P...",1.3662,False
1,cluster_2,"Age, fT3, Low_T3",1.1566,True
2,cluster_1,"TSH, fT4, SCH, SCT, Hyperthyroid",1.0194,True
3,cluster_7,Hypertension,1.0150,False
4,cluster_3,Acute_MI,1.0092,False
5,cluster_6,Diabetes,1.0060,False
6,cluster_4,Hypothyroid,1.0000,True


### Ablation (multi-variable, clustered) — CV17_THY_CONT_STATES, strict 10y

,feature,members,importance,is_thyroid
0,cluster_3,"Age, fT3, Low_T3",1.0581,True
1,cluster_5,"Angina, Previous_CABG, Previous_PCI, Previous_...",1.0365,False
2,cluster_6,"Diabetes, Hypertension",1.0126,False
3,cluster_4,Hypothyroid,1.0024,True
4,cluster_2,"Gender, Smoke",1.0000,False
5,cluster_7,Acute_MI,0.9982,False
6,cluster_1,"TSH, fT4, SCH, SCT, Hyperthyroid",0.9811,True


In [18]:
# Where do thyroid features rank across ALL sets (single-variable)?
def thyroid_rank_summary(horizon):
    rows = []
    for fs in abl["feature_set"].unique():
        s = abl[(abl["horizon"] == horizon) & (abl["feature_set"] == fs)
                & (abl["mode"] == "single")].copy()
        if s.empty:
            continue
        s = s.sort_values("importance", ascending=False).reset_index(drop=True)
        s["rank"] = s.index + 1
        thy = s[s["is_thyroid"]]
        for _, r in thy.iterrows():
            rows.append({"feature_set": fs, "feature": r["feature"],
                         "rank": int(r["rank"]), "of": len(s),
                         "importance": round(r["importance"], 4)})
    return pd.DataFrame(rows)

for h in HORIZONS:
    display(Markdown(f"### Thyroid-feature placement in ablation ranking — strict {h}y"))
    display(thyroid_rank_summary(h))


### Thyroid-feature placement in ablation ranking — strict 7y

,feature_set,feature,rank,of,importance
0,CV17_THY_CONT_STATES,TSH,4,25,1.0194
1,CV17_THY_CONT_STATES,Low_T3,12,25,1.0074
2,CV17_THY_CONT_STATES,SCT,18,25,1.0000
3,CV17_THY_CONT_STATES,Hypothyroid,19,25,1.0000
4,CV17_THY_CONT_STATES,fT3,20,25,0.9985
5,CV17_THY_CONT_STATES,Hyperthyroid,21,25,0.9964
6,CV17_THY_CONT_STATES,SCH,23,25,0.9930
7,CV17_THY_CONT_STATES,fT4,24,25,0.9929
8,CV17_THY_ABNORMAL_BIN,Thyroid_abnormal,7,18,1.0277
9,CV17_THY_STATE_ORD,thyroid_ord,17,18,1.0000


### Thyroid-feature placement in ablation ranking — strict 10y

,feature_set,feature,rank,of,importance
0,CV17_THY_CONT_STATES,Hypothyroid,6,25,1.0024
1,CV17_THY_CONT_STATES,SCH,10,25,1.0000
2,CV17_THY_CONT_STATES,Hyperthyroid,11,25,1.0000
3,CV17_THY_CONT_STATES,Low_T3,12,25,0.9994
4,CV17_THY_CONT_STATES,SCT,18,25,0.9958
5,CV17_THY_CONT_STATES,TSH,21,25,0.9934
6,CV17_THY_CONT_STATES,fT3,22,25,0.9893
7,CV17_THY_CONT_STATES,fT4,23,25,0.9887
8,CV17_THY_ABNORMAL_BIN,Thyroid_abnormal,10,18,1.0094
9,CV17_THY_STATE_ORD,thyroid_ord,14,18,1.0024


In [19]:
# Ablation for the WINNER ensemble: thyroid placement in CV17_THY_CONT_STATES.
abl_w = RA.build_ablation(cohorts, horizons=HORIZONS, model_name=WINNER,
                          seed=SEED, force=FORCE, out_suffix=WINNER)

def thyroid_rank_summary_df(frame, horizon, feature_set="CV17_THY_CONT_STATES"):
    s = frame[(frame["horizon"] == horizon) &
              (frame["feature_set"] == feature_set) &
              (frame["mode"] == "single")].copy()
    s = s.sort_values("importance", ascending=False).reset_index(drop=True)
    s["rank"] = s.index + 1
    return s[s["is_thyroid"]][["rank", "feature", "importance"]]

for h in HORIZONS:
    display(Markdown(f"### Thyroid placement in CV17_THY_CONT_STATES ablation — strict {h}y "
                     f"(WINNER: {WINNER})"))
    display(thyroid_rank_summary_df(abl_w, h).round(4).reset_index(drop=True))


### Thyroid placement in CV17_THY_CONT_STATES ablation — strict 7y (WINNER: ENS_STACK)

,rank,feature,importance
0,10,TSH,1.0132
1,15,Low_T3,1.0054
2,16,SCH,1.0036
3,19,SCT,1.0000
4,20,Hypothyroid,1.0000
5,22,Hyperthyroid,0.9983
6,24,fT4,0.9931
7,25,fT3,0.9775


### Thyroid placement in CV17_THY_CONT_STATES ablation — strict 10y (WINNER: ENS_STACK)

,rank,feature,importance
0,9,SCH,1.0000
1,10,Hypothyroid,1.0000
2,11,Hyperthyroid,1.0000
3,17,fT3,0.9976
4,18,SCT,0.9976
5,19,Low_T3,0.9952
6,24,TSH,0.9888
7,25,fT4,0.9857


## Sensitivity S5 — fixed-parameter calibration / Brier

Brier scores and reliability curves from the existing fixed-parameter models
are retained as a pre-tuning sensitivity analysis. The primary held-out
calibration and decision-curve analyses for the tuned paper ensemble are
reported separately below when a matching tuning cache is available.


In [20]:
# Brier is already in the classification table; pivot it.
brier = clf.pivot_table(index=["horizon", "feature_set"],
                        columns=["model", "scheme"], values="brier")
display(Markdown("### Brier score by model / scheme (lower is better)"))
display(brier.round(3))


### Brier score by model / scheme (lower is better)

model                     AdaBoost              ENSEMBLE             ENS_STACK            ENS_STACK_GB            ENS_VOTE_BOOST            ENS_VOTE_DIVERSE  \
scheme                  A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20   A_5fold_cv B_60_20_20     A_5fold_cv B_60_20_20       A_5fold_cv   
horizon feature_set                                                                                                                                            
7       CV17                 0.196      0.197      0.142      0.141      0.111      0.108        0.111      0.108          0.150      0.145            0.134   
        CV17_THY_ABNORMAL_BIN             0.197      0.196      0.142      0.140      0.111      0.109        0.111      0.108          0.150      0.145            0.132   
        CV17_THY_STATES             0.196      0.194      0.142      0.140      0.111      0.109        0.111      0.108          0.150      0.145            0.132   
        CV17_THY_CONT            0.195      0.194      0.141      0.140      0.110      0.106        0.110      0.106          0.146      0.141            0.130   
        CV17_THY_STATE_ORD             0.196      0.195      0.141      0.140      0.111      0.108        0.111      0.108          0.150      0.146            0.132   
        CV17_THY_CONT_STATES_RATIO           0.196      0.198      0.142      0.140      0.110      0.106        0.110      0.106          0.146      0.140            0.129   
        CV17_THY_CONT_RATIO      0.196      0.198      0.141      0.140      0.109      0.105        0.109      0.106          0.145      0.140            0.129   
        CV17_THY_CONT_STATES           0.195      0.194      0.141      0.139      0.110      0.107        0.110      0.107          0.146      0.141            0.129   
10      CV17                 0.193      0.190      0.156      0.159      0.147      0.152        0.147      0.152          0.158      0.162            0.152   
        CV17_THY_ABNORMAL_BIN             0.193      0.191      0.156      0.158      0.147      0.151        0.147      0.151          0.158      0.159            0.151   
        CV17_THY_STATES             0.194      0.190      0.156      0.158      0.146      0.152        0.146      0.152          0.158      0.161            0.151   
        CV17_THY_CONT            0.193      0.193      0.156      0.159      0.146      0.151        0.145      0.151          0.157      0.160            0.150   
        CV17_THY_STATE_ORD             0.193      0.190      0.156      0.159      0.147      0.152        0.146      0.152          0.159      0.163            0.151   
        CV17_THY_CONT_STATES_RATIO           0.193      0.193      0.156      0.159      0.146      0.151        0.145      0.151          0.156      0.157            0.149   
        CV17_THY_CONT_RATIO      0.193      0.193      0.156      0.159      0.146      0.150        0.145      0.150          0.156      0.157            0.149   
        CV17_THY_CONT_STATES           0.193      0.193      0.156      0.159      0.146      0.151        0.146      0.151          0.156      0.160            0.149   

model                              ENS_VOTE_WIDE            ENS_VOTE_XGB            GradientBoosting            KNeighbors            LogisticRegression  \
scheme                  B_60_20_20    A_5fold_cv B_60_20_20   A_5fold_cv B_60_20_20       A_5fold_cv B_60_20_20 A_5fold_cv B_60_20_20         A_5fold_cv   
horizon feature_set                                                                                                                                        
7       CV17                 0.129         0.132      0.133        0.131      0.127            0.152      0.149      0.216      0.199              0.162   
        CV17_THY_ABNORMAL_BIN             0.127         0.129      0.124        0.131      0.126            0.153      0.147      0.211      0.200              0.162   
        CV17_THY_STATES             0.

In [21]:
# Reliability curves: ensemble + members on CV17 @ 7y (scheme A OOF).
from src.alignment.evaluation import evaluate_cv
from src.alignment.calibration import plot_calibration

H = 7
X, y = extract_Xy_align(cohorts[H], "CV17", f"y{H}")
y_arr = np.asarray(pd.Series(np.asarray(y)))
curves = []
for m in ["ENSEMBLE", WINNER] + ENSEMBLE_MEMBERS:
    r = evaluate_cv(X, y, m, seed=SEED)
    curves.append((m, y_arr, r["oof_proba"]))
    if m == "ENSEMBLE":
        paper_r = r

fig, ax = plt.subplots(figsize=(7, 6))
plot_calibration(ax, curves,
                 f"Calibration — CV17, strict {H}y (scheme A OOF)\n"
                 f"paper ENSEMBLE vs WINNER ({WINNER})")
plt.show()

# Paper-like check: predicted survival probability against observed event time.
cal = cohorts[H][["time_years", f"y{H}"]].copy()
cal["p_survive"] = 1.0 - paper_r["oof_proba"]
event_bins = pd.cut(cal["time_years"], bins=[0, 2, 4, H],
                    labels=["CD 0–2y", "CD 2–4y", "CD 4–7y"],
                    include_lowest=True)
cal["observed_group"] = np.where(cal[f"y{H}"] == 0,
                                  f"survived ≥{H}y", event_bins.astype(str))
order = ["CD 0–2y", "CD 2–4y", "CD 4–7y", f"survived ≥{H}y"]
cal["observed_group"] = pd.Categorical(
    cal["observed_group"], categories=order, ordered=True)
fig, ax = plt.subplots(figsize=(9, 5))
cal.boxplot(column="p_survive", by="observed_group", ax=ax, grid=False)
ax.set_title("Predicted survival vs observed outcome/time — ENSEMBLE OOF")
ax.set_xlabel("Observed group"); ax.set_ylabel("Predicted P(survive ≥7y)")
plt.suptitle(""); plt.xticks(rotation=20); plt.tight_layout(); plt.show()


## 6. Held-out clinical utility of the tuned paper ensemble

These analyses use only the **paired held-out test predictions** persisted by
the extended-tuning pipeline. CV17 and CV17_THY_CONT_STATES are compared patient by
patient. Risk thresholds (10%, 20%, 30%, 40%, 50%) are prespecified scenarios:
they are never selected or optimised on test outcomes.

- Calibration: Brier, intercept/slope, ECE and reliability bins.
- Decision-curve analysis (DCA): net benefit vs treat-all/treat-none and paired
  bootstrap Δ net benefit (thyroid − CV17).
- Clinical-impact scenarios: flagged patients, events detected/missed and false
  positives per 1,000.

> These are **decision-analytic proxies**, not evidence that deployment changes
> care or improves realised patient outcomes. Prospective validation in the
> intended workflow remains necessary.


In [22]:
clinical_calibration_rows = []
clinical_calibration_curves = []
clinical_dca_frames = []
clinical_impact_frames = []

if paper_tuning_run.status in {"computed", "cache_complete"}:
    for h in HORIZONS:
        base_artifact = paper_tuning_run.get_artifact(
            h, "CV17", "ENSEMBLE_PAPER", stage="ensemble")
        thyroid_artifact = paper_tuning_run.get_artifact(
            h, "CV17_THY_CONT_STATES", "ENSEMBLE_PAPER", stage="ensemble")

        base_pred = pd.read_csv(base_artifact["predictions_csv"])
        thyroid_pred = pd.read_csv(thyroid_artifact["predictions_csv"])
        base_test = base_pred[base_pred["split"] == "test"][
            ["row_position", "row_index", "y", "p_event"]
        ].rename(columns={"p_event": "p_event_cv17"})
        thyroid_test = thyroid_pred[thyroid_pred["split"] == "test"][
            ["row_position", "row_index", "y", "p_event"]
        ].rename(columns={"p_event": "p_event_thyroid"})
        base_test = base_test.reset_index(drop=True)
        thyroid_test = thyroid_test.reset_index(drop=True)
        pairing_columns = ["row_position", "row_index", "y"]
        if (
            len(base_test) != len(thyroid_test)
            or base_test["row_position"].duplicated().any()
            or thyroid_test["row_position"].duplicated().any()
            or base_test["row_index"].duplicated().any()
            or thyroid_test["row_index"].duplicated().any()
            or not base_test[pairing_columns].equals(
                thyroid_test[pairing_columns])
        ):
            raise RuntimeError(
                f"Unpaired or differently ordered test predictions "
                f"at {h} years")
        paired = base_test.copy()
        paired["p_event_thyroid"] = (
            thyroid_test["p_event_thyroid"].to_numpy())

        y_test = paired["y"].to_numpy()
        p_base = paired["p_event_cv17"].to_numpy()
        p_thyroid = paired["p_event_thyroid"].to_numpy()

        for feature_set, probabilities in (
            ("CV17", p_base), ("CV17_THY_CONT_STATES", p_thyroid)
        ):
            summary = calibration_summary(
                y_test, probabilities, n_bins=CALIBRATION_BINS,
                strategy="quantile")
            clinical_calibration_rows.append({
                "horizon": h, "feature_set": feature_set, **summary})
            curve = calibration_curve_table(
                y_test, probabilities, n_bins=CALIBRATION_BINS,
                strategy="quantile")
            curve.insert(0, "feature_set", feature_set)
            curve.insert(0, "horizon", h)
            clinical_calibration_curves.append(curve)

        dca = decision_curve(
            y_test,
            p_thyroid,
            CLINICAL_RISK_THRESHOLDS,
            y_proba_comparator=p_base,
            n_boot=N_BOOT,
            seed=SEED,
        )
        dca.insert(0, "horizon", h)
        clinical_dca_frames.append(dca)

        impact_base = clinical_impact_table(
            y_test, p_base, CLINICAL_RISK_THRESHOLDS)
        impact_thyroid = clinical_impact_table(
            y_test, p_thyroid, CLINICAL_RISK_THRESHOLDS)
        impact = impact_base.merge(
            impact_thyroid, on="threshold", suffixes=("_cv17", "_thyroid"),
            validate="one_to_one")
        for metric in (
            "high_risk_per_1000", "events_detected_per_1000",
            "events_missed_per_1000", "fp_per_1000", "net_benefit",
        ):
            impact[f"delta_{metric}"] = (
                impact[f"{metric}_thyroid"] - impact[f"{metric}_cv17"])
        impact.insert(0, "horizon", h)
        clinical_impact_frames.append(impact)

clinical_calibration = pd.DataFrame(clinical_calibration_rows)
clinical_calibration_curve = (
    pd.concat(clinical_calibration_curves, ignore_index=True)
    if clinical_calibration_curves else pd.DataFrame())
clinical_dca = (
    pd.concat(clinical_dca_frames, ignore_index=True)
    if clinical_dca_frames else pd.DataFrame())
clinical_impact = (
    pd.concat(clinical_impact_frames, ignore_index=True)
    if clinical_impact_frames else pd.DataFrame())

if clinical_calibration.empty:
    display(Markdown(
        "Clinical-utility tables are unavailable because the matching tuned "
        "ensemble cache is absent. No model fitting was triggered."))
else:
    display(Markdown("### Held-out calibration summary"))
    display(clinical_calibration.round(4))
    display(Markdown("### Paired decision-curve analysis"))
    display(clinical_dca.round(4))
    display(Markdown("### Clinical-impact scenarios (thyroid minus CV17 deltas)"))
    impact_cols = [
        "horizon", "threshold",
        "high_risk_per_1000_cv17", "high_risk_per_1000_thyroid",
        "delta_high_risk_per_1000",
        "events_detected_per_1000_cv17",
        "events_detected_per_1000_thyroid",
        "delta_events_detected_per_1000",
        "events_missed_per_1000_cv17",
        "events_missed_per_1000_thyroid",
        "delta_events_missed_per_1000",
        "fp_per_1000_cv17", "fp_per_1000_thyroid",
        "delta_fp_per_1000", "delta_net_benefit",
    ]
    display(clinical_impact[impact_cols].round(3))


### Held-out calibration summary

,horizon,feature_set,n,observed_events,expected_events,brier,calibration_intercept,calibration_slope,recalibration_status,ece,observed_rate,expected_rate,observed_to_expected_ratio
0,7,CV17,878,168,318.1578,0.1414,-0.9807,1.5965,ok,0.1710,0.1913,0.3624,0.5280
1,7,CV17_THY_CONT_STATES,878,168,313.2946,0.1385,-0.9790,1.5269,ok,0.1655,0.1913,0.3568,0.5362
2,10,CV17,523,192,243.2486,0.1593,-0.6081,1.3885,ok,0.1162,0.3671,0.4651,0.7893
3,10,CV17_THY_CONT_STATES,523,192,240.8479,0.1599,-0.5608,1.5087,ok,0.1091,0.3671,0.4605,0.7972


### Paired decision-curve analysis

,horizon,threshold,net_benefit_model,net_benefit_treat_all,net_benefit_treat_none,net_benefit_comparator,delta_net_benefit,delta_ci_low,delta_ci_high,n_bootstrap
0,7,0.1,0.1015,0.1015,0.0,0.1015,0.0000,0.0000,0.0000,1000
1,7,0.2,0.0521,-0.0108,0.0,0.0490,0.0031,-0.0026,0.0077,1000
2,7,0.3,0.0267,-0.1552,0.0,0.0238,0.0029,-0.0075,0.0130,1000
3,7,0.4,0.0182,-0.3478,0.0,0.0121,0.0061,-0.0057,0.0190,1000
4,7,0.5,0.0171,-0.6173,0.0,0.0114,0.0057,-0.0080,0.0205,1000
5,10,0.1,0.2968,0.2968,0.0,0.2981,-0.0013,-0.0023,-0.0004,1000
6,10,0.2,0.2467,0.2089,0.0,0.2524,-0.0057,-0.0105,-0.0014,1000
7,10,0.3,0.2128,0.0959,0.0,0.2081,0.0046,-0.0066,0.0161,1000
8,10,0.4,0.1804,-0.0548,0.0,0.1695,0.0108,-0.0019,0.0242,1000
9,10,0.5,0.1319,-0.2658,0.0,0.1358,-0.0038,-0.0229,0.0134,1000


### Clinical-impact scenarios (thyroid minus CV17 deltas)

,horizon,threshold,high_risk_per_1000_cv17,high_risk_per_1000_thyroid,delta_high_risk_per_1000,events_detected_per_1000_cv17,events_detected_per_1000_thyroid,delta_events_detected_per_1000,events_missed_per_1000_cv17,events_missed_per_1000_thyroid,delta_events_missed_per_1000,fp_per_1000_cv17,fp_per_1000_thyroid,delta_fp_per_1000,delta_net_benefit
0,7,0.1,1000.000,1000.000,0.000,191.344,191.344,0.000,0.000,0.000,0.000,808.656,808.656,0.000,0.000
1,7,0.2,715.262,691.344,-23.918,182.232,179.954,-2.278,9.112,11.390,2.278,533.030,511.390,-21.640,0.003
2,7,0.3,510.251,492.027,-18.223,169.704,166.287,-3.417,21.640,25.057,3.417,340.547,325.740,-14.806,0.003
3,7,0.4,369.021,348.519,-20.501,154.897,150.342,-4.556,36.446,41.002,4.556,214.123,198.178,-15.945,0.006
4,7,0.5,250.569,260.820,10.251,130.979,138.952,7.973,60.364,52.392,-7.973,119.590,121.868,2.278,0.006
5,10,0.1,988.528,1000.000,11.472,367.113,367.113,0.000,0.000,0.000,0.000,621.415,632.887,11.472,-0.001
6,10,0.2,787.763,810.707,22.945,359.465,359.465,0.000,7.648,7.648,0.000,428.298,451.243,22.945,-0.006
7,10,0.3,642.447,663.480,21.033,338.432,347.992,9.560,28.681,19.120,-9.560,304.015,315.488,11.472,0.005
8,10,0.4,548.757,537.285,-11.472,321.224,323.136,1.912,45.889,43.977,-1.912,227.533,214.149,-13.384,0.011
9,10,0.5,460.803,453.155,-7.648,298.279,292.543,-5.736,68.834,74.570,5.736,162.524,160.612,-1.912,-0.004


In [23]:
if not clinical_calibration_curve.empty:
    fig, axes = plt.subplots(len(HORIZONS), 2, figsize=(13, 5 * len(HORIZONS)))
    axes = np.atleast_2d(axes)
    for row, h in enumerate(HORIZONS):
        ax_cal, ax_dca = axes[row]
        curve_h = clinical_calibration_curve[
            clinical_calibration_curve["horizon"] == h]
        for feature_set, marker in (("CV17", "o-"), ("CV17_THY_CONT_STATES", "s-")):
            d = curve_h[curve_h["feature_set"] == feature_set]
            ax_cal.plot(d["mean_predicted"], d["observed_rate"], marker,
                        label=feature_set)
        ax_cal.plot([0, 1], [0, 1], "k--", linewidth=1, label="ideal")
        ax_cal.set(xlabel="Predicted event risk", ylabel="Observed event rate",
                   title=f"Held-out calibration — {h}y")
        ax_cal.grid(alpha=0.25); ax_cal.legend()

        d = clinical_dca[clinical_dca["horizon"] == h]
        ax_dca.plot(d["threshold"], d["net_benefit_comparator"], "o-",
                    label="CV17")
        ax_dca.plot(d["threshold"], d["net_benefit_model"], "s-",
                    label="CV17_THY_CONT_STATES")
        ax_dca.plot(d["threshold"], d["net_benefit_treat_all"], "--",
                    label="treat all")
        ax_dca.plot(d["threshold"], d["net_benefit_treat_none"], ":",
                    label="treat none")
        ax_dca.set(xlabel="Prespecified event-risk threshold",
                   ylabel="Net benefit", title=f"Decision curve — {h}y")
        ax_dca.grid(alpha=0.25); ax_dca.legend()
    fig.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6.5 * len(HORIZONS), 4.5),
                             squeeze=False)
    for ax, h in zip(axes[0], HORIZONS):
        d = clinical_dca[clinical_dca["horizon"] == h]
        ax.axhline(0, color="black", linewidth=1)
        ax.plot(d["threshold"], d["delta_net_benefit"], "o-",
                label="thyroid − CV17")
        ax.fill_between(d["threshold"], d["delta_ci_low"], d["delta_ci_high"],
                        alpha=0.2, label="paired bootstrap 95% CI")
        ax.set(xlabel="Prespecified event-risk threshold",
               ylabel="Δ net benefit", title=f"Incremental DCA — {h}y")
        ax.grid(alpha=0.25); ax.legend()
    fig.tight_layout(); plt.show()


## 7. Differences vs the paper

Declared differences (scope: **retrace the steps, not the numbers**).


In [24]:
diff = pd.DataFrame([
    ["Classification cohort", "3987 (757 CD)",
     f"strict {len(cohorts[7])} ({int(cohorts[7]['y7'].sum())} CVD ≤7y)"],
    ["Creatinine / eGFR", "included", "excluded (too many missing)"],
    ["Primary validation", "60/20/20; internal 2-fold tuning CV",
     "same scheme; paired CV17 vs thyroid test rows"],
    ["Hyperparameter tuning", "5000-iteration random search",
     "5000 draws/model/set/horizon; F1-macro; training only"],
    ["Fixed-parameter analyses", "—",
     "retained and explicitly labelled pre-tuning sensitivity"],
    ["Missing values", "replace missing with 0 before scaling",
     "median imputation fitted inside each training fold"],
    ["Sampling", "under + SMOTE family selected on validation",
     "optional post-search refinement; validation only; test untouched"],
    ["Decision threshold", "0.5 for the published classifier",
     "0.5 for primary tuned test metrics; clinical scenarios prespecified"],
    ["Non-CVD deaths", "excluded per tutto il follow-up",
     "per scelta, survivor se la morte è oltre l'orizzonte"],
    ["Thyroid-state encoding", "—", "5 dummies (eutiroideo = reference)"],
    ["Clinical utility", "calibration/Brier",
     "calibration + DCA + impact scenarios, decision-analytic only"],
    ["Horizons", "7 years; 10-year sensitivity", "7 and 10 years"],
    ["Objective", "predict cardiac death",
     "incremental value of thyroid over CV17"],
], columns=["Aspect", "Paper (Pingitore 2024)", "This thesis"])
display(Markdown("### Declared differences vs the paper"))
display(diff)


### Declared differences vs the paper

,Aspect,Paper (Pingitore 2024),This thesis
0,Classification cohort,3987 (757 CD),strict 4390 (843 CVD ≤7y)
1,Creatinine / eGFR,included,excluded (too many missing)
2,Primary validation,60/20/20; internal 2-fold tuning CV,same scheme; paired CV17 vs thyroid test rows
3,Hyperparameter tuning,5000-iteration random search,5000 draws/model/set/horizon; F1-macro; traini...
4,Fixed-parameter analyses,—,retained and explicitly labelled pre-tuning se...
5,Missing values,replace missing with 0 before scaling,median imputation fitted inside each training ...
6,Sampling,under + SMOTE family selected on validation,optional post-search refinement; validation on...
7,Decision threshold,0.5 for the published classifier,0.5 for primary tuned test metrics; clinical s...
8,Non-CVD deaths,excluded per tutto il follow-up,"per scelta, survivor se la morte è oltre l'ori..."
9,Thyroid-state encoding,—,5 dummies (eutiroideo = reference)


## 8. Final synthesis

The tuned scheme-B ensemble is the primary paper-like analysis when its cache
is available. The previously computed A/B panels, survival indicator and
ablation are retained as explicitly labelled **pre-tuning sensitivities**.

<!-- paper-tuning-postrun:begin -->
### Complete tuned held-out results

Cache-complete primary run: `3fc4b61a32467cfb277ceb1a5a9b670b9620537365426b93af2605e415c69f09`. The values below are generated from the persisted held-out test predictions; no test threshold was optimised.

- **7-year held-out test:** F1-macro 0.739 (CV17) vs 0.751 (CV17+thyroid), paired Δ=+0.012 [-0.005, +0.032]; AUROC 0.841 vs 0.843, paired Δ=+0.002 [-0.007, +0.011]. the paired 95% bootstrap CI for ΔF1 includes 0; the paired 95% bootstrap CI for ΔAUROC includes 0.
  Calibration: Brier 0.141 vs 0.138 (Δ thyroid−CV17 -0.003); ECE 0.171 vs 0.165. DCA Δ net benefit is positive at 20%, 30%, 40%, 50%; its paired CI is entirely above 0 at none and entirely below 0 at none.
  Across the prespecified 10–50% scenarios, adding thyroid changes detected events by -4.6 to +8.0 per 1,000 and false positives by -21.6 to +2.3 per 1,000.

- **10-year held-out test:** F1-macro 0.762 (CV17) vs 0.757 (CV17+thyroid), paired Δ=-0.005 [-0.023, +0.012]; AUROC 0.856 vs 0.856, paired Δ=+0.000 [-0.007, +0.007]. the paired 95% bootstrap CI for ΔF1 includes 0; the paired 95% bootstrap CI for ΔAUROC includes 0.
  Calibration: Brier 0.159 vs 0.160 (Δ thyroid−CV17 +0.001); ECE 0.116 vs 0.109. DCA Δ net benefit is positive at 30%, 40%; its paired CI is entirely above 0 at none and entirely below 0 at 10%, 20%.
  Across the prespecified 10–50% scenarios, adding thyroid changes detected events by -5.7 to +9.6 per 1,000 and false positives by -13.4 to +22.9 per 1,000.

The DCA and clinical-impact quantities are decision-analytic scenarios, not evidence of realised benefit after deployment. Calibration summaries are descriptive and have no confidence intervals. No multiplicity correction was applied, so the paired intervals and threshold-wise DCA comparisons remain exploratory. The fixed-parameter panels below/above remain sensitivity analyses and do not replace this tuned held-out comparison.
<!-- paper-tuning-postrun:end -->


In [25]:
lines = []

if paper_tuning_run.status in {"computed", "cache_complete"}:
    tuned_test = paper_tuning_run.ensemble_results.query("split == 'test'")
    for h in HORIZONS:
        d = tuned_test[tuned_test["horizon"] == h]
        if len(d):
            base = d[d["feature_set"] == "CV17"].iloc[0]
            thy = d[d["feature_set"] == "CV17_THY_CONT_STATES"].iloc[0]
            lines.append(
                f"**Primary tuned {h}y test** — CV17 F1-macro={base.f1_macro:.3f}, "
                f"CV17_THY_CONT_STATES={thy.f1_macro:.3f}; AUROC {base.roc_auc:.3f} vs "
                f"{thy.roc_auc:.3f}.")
    if not paper_tuning_run.incremental_results.empty:
        for r in paper_tuning_run.incremental_results.itertuples():
            lines.append(
                f"**Primary tuned {int(r.horizon)}y paired test** — "
                f"ΔF1={r.delta_f1_macro:+.3f} "
                f"[{r.delta_f1_ci_lo:+.3f}, {r.delta_f1_ci_hi:+.3f}]; "
                f"ΔAUROC={r.delta_auroc:+.3f} "
                f"[{r.delta_auroc_ci_lo:+.3f}, {r.delta_auroc_ci_hi:+.3f}].")
else:
    lines.append(
        "**Primary tuned analysis not displayed:** matching extended-tuning "
        "cache absent. No 5,000-draw search was launched automatically.")

lines.append("**Pre-tuning sensitivities (context, not primary tuned results):**")

# Exploratory ensemble selection summary: CV only, never holdout test.
cv17_ens = clf[(clf["model"].isin(ENSEMBLE_NAMES)) &
                 (clf["feature_set"] == "CV17") &
                 (clf["scheme"] == "A_5fold_cv")]
ens_mean = cv17_ens.groupby("model")["f1_macro"].mean()
paper_f1 = ens_mean.get("ENSEMBLE", float("nan"))
win_f1 = ens_mean.get(WINNER, float("nan"))
lines.append(f"**Exploratory CV-only ensemble selection** — best alternative = **{WINNER}** "
             f"(mean CV17 F1-macro = {win_f1:.3f}) vs paper ENSEMBLE "
             f"({paper_f1:.3f}); Δ = {win_f1 - paper_f1:+.3f}. "
             + "This comparison is a pre-tuning sensitivity analysis.")

ens = clf[clf["model"] == "ENSEMBLE"]
for h in HORIZONS:
    for scheme in ["A_5fold_cv", "B_60_20_20"]:
        e = ens[(ens["horizon"] == h) & (ens["scheme"] == scheme)]
        base = e[e["feature_set"] == "CV17"]["f1_macro"].values
        base = base[0] if len(base) else float("nan")
        best = e.sort_values("f1_macro", ascending=False).iloc[0]
        lines.append(f"**{h}y · {scheme}** — CV17 ensemble F1-macro = {base:.3f}; "
                     f"best set = {best['feature_set']} ({best['f1_macro']:.3f}).")

        # any thyroid set with ΔF1-macro CI excluding 0?
        di = incr[(incr["horizon"] == h) & (incr["scheme"] == scheme)]
        sig = di[(di["delta_f1_ci_lo"] > 0) | (di["delta_f1_ci_hi"] < 0)]
        if len(sig):
            ss = ", ".join(f"{r.feature_set} (ΔF1={r.delta_f1_macro:+.3f})"
                           for r in sig.itertuples())
            lines.append(f"&nbsp;&nbsp;• thyroid sets with ΔF1 CI excluding 0: {ss}")
        else:
            lines.append("&nbsp;&nbsp;• no thyroid set has a ΔF1-macro CI excluding 0.")

# Δ C-index
for h in HORIZONS:
    for scheme in ["A", "B"]:
        d = cindex_df[(cindex_df["horizon"] == h) &
                      (cindex_df["scheme"] == scheme) &
                      (cindex_df["feature_set"].str.startswith("DELTA"))]
        if len(d):
            r = d.iloc[0]
            lines.append(f"**{h}y · scheme {scheme}** — ML-indicator "
                         f"Δ C-index (CV17_THY_CONT_STATES − CV17) = {r['c_index']:+.4f} "
                         f"[{r['c_index_ci_lo']:+.4f}, {r['c_index_ci_hi']:+.4f}].")

# ablation placement
for h in HORIZONS:
    tr = thyroid_rank_summary(h)
    thy26 = tr[tr["feature_set"] == "CV17_THY_CONT_STATES"]
    if len(thy26):
        best = thy26.sort_values("rank").iloc[0]
        lines.append(f"**{h}y** — best-ranked thyroid feature in CV17_THY_CONT_STATES "
                     f"ablation: {best['feature']} (rank {best['rank']}/{best['of']}).")

display(Markdown("### Primary tuned result + pre-tuning sensitivities\n\n" + "\n\n".join(lines)))


### Primary tuned result + pre-tuning sensitivities

**Primary tuned 7y test** — CV17 F1-macro=0.739, CV17_THY_CONT_STATES=0.751; AUROC 0.841 vs 0.843.

**Primary tuned 10y test** — CV17 F1-macro=0.762, CV17_THY_CONT_STATES=0.757; AUROC 0.856 vs 0.856.

**Primary tuned 7y paired test** — ΔF1=+0.012 [-0.005, +0.032]; ΔAUROC=+0.002 [-0.007, +0.011].

**Primary tuned 10y paired test** — ΔF1=-0.005 [-0.023, +0.012]; ΔAUROC=+0.000 [-0.007, +0.007].

**Pre-tuning sensitivities (context, not primary tuned results):**

**Exploratory CV-only ensemble selection** — best alternative = **ENS_STACK** (mean CV17 F1-macro = 0.748) vs paper ENSEMBLE (0.749); Δ = -0.001. This comparison is a pre-tuning sensitivity analysis.

**7y · A_5fold_cv** — CV17 ensemble F1-macro = 0.731; best set = CV17_THY_CONT (0.736).

&nbsp;&nbsp;• no thyroid set has a ΔF1-macro CI excluding 0.

**7y · B_60_20_20** — CV17 ensemble F1-macro = 0.748; best set = CV17_THY_STATE_ORD (0.763).

&nbsp;&nbsp;• no thyroid set has a ΔF1-macro CI excluding 0.

**10y · A_5fold_cv** — CV17 ensemble F1-macro = 0.768; best set = CV17_THY_CONT_STATES_RATIO (0.773).

&nbsp;&nbsp;• no thyroid set has a ΔF1-macro CI excluding 0.

**10y · B_60_20_20** — CV17 ensemble F1-macro = 0.753; best set = CV17_THY_ABNORMAL_BIN (0.763).

&nbsp;&nbsp;• no thyroid set has a ΔF1-macro CI excluding 0.

**7y · scheme A** — ML-indicator Δ C-index (CV17_THY_CONT_STATES − CV17) = +0.0039 [+0.0005, +0.0073].

**7y · scheme B** — ML-indicator Δ C-index (CV17_THY_CONT_STATES − CV17) = +0.0032 [-0.0044, +0.0108].

**10y · scheme A** — ML-indicator Δ C-index (CV17_THY_CONT_STATES − CV17) = +0.0024 [-0.0006, +0.0056].

**10y · scheme B** — ML-indicator Δ C-index (CV17_THY_CONT_STATES − CV17) = +0.0038 [-0.0029, +0.0096].

**7y** — best-ranked thyroid feature in CV17_THY_CONT_STATES ablation: TSH (rank 4/25).

**10y** — best-ranked thyroid feature in CV17_THY_CONT_STATES ablation: Hypothyroid (rank 6/25).

---
*Cohort: strict. Horizons: 7 & 10 years. The primary paper-like analysis uses
scheme B (60/20/20), 5,000 random-search draws and two-fold CV within training.
Fixed-parameter schemes A/B are reported as pre-tuning sensitivities. Cache
reuse requires matching data, features, protocol, configuration and dependency
versions. Decision-curve and impact tables are decision-analytic scenarios,
not evidence of realised patient benefit.*
